In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os
import zipfile

zip_path = '/content/drive/MyDrive/ML_Skin_Cancer/archive.zip'
extract_path = '/content/dataset'
yolo_dataset_path = '/content/yolo_dataset'  

if not os.path.exists(extract_path):
    print("Mengekstrak dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Ekstraksi selesai.")
else:
    print("Dataset sudah diekstrak sebelumnya.")

In [ ]:
import pandas as pd

metadata_path = f'{extract_path}/HAM10000_metadata.csv'
df = pd.read_csv(metadata_path)

classes = ['nv', 'mel', 'bkl', 'bcc', 'akiec', 'vasc', 'df']
label2idx = {c: i for i, c in enumerate(classes)}
df['label'] = df['dx'].map(label2idx)

print(f"Dimensi DataFrame Metadata: {df.shape}")
df.head()

(10015, 8)


,lesion_id,image_id,dx,dx_type,age,sex,localization,label
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,2
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,2
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,2
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,2
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,2


In [ ]:
import glob

img_dir1 = f'{extract_path}/HAM10000_images_part_1'
img_dir2 = f'{extract_path}/HAM10000_images_part_2'

all_image_paths = {}
for d in [img_dir1, img_dir2]:
    for f in glob.glob(os.path.join(d, '*.jpg')):
        image_id = os.path.basename(f).replace('.jpg', '')
        all_image_paths[image_id] = f

print(f"Total gambar ditemukan: {len(all_image_paths)}")

Total gambar: 10015


In [ ]:
from sklearn.model_selection import train_test_split

# Membagi dengan stratifikasi agar distribusi kelas seimbang di setiap split
train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

print(f"Jumlah Data -> Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

Train: 7010, Val: 1502, Test: 1503


# SILAKAN GANTI ANGKA DI BAWAH INI UNTUK MEMILIH SKENARIO:
- 1 = Tanpa Preprocessing (Baseline)
- 2 = WRS (Oversampling) Saja
- 3 = DullRazor Saja
- 4 = CLAHE Saja
- 5 = WRS + DullRazor + CLAHE

In [ ]:
import os
import shutil
import cv2
import pandas as pd
import numpy as np

SKENARIO = 5

def apply_dullrazor(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_CROSS, (9,9))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    bhg = cv2.GaussianBlur(blackhat, (3,3), cv2.BORDER_DEFAULT)
    ret, mask = cv2.threshold(bhg, 10, 255, cv2.THRESH_BINARY)
    dst = cv2.inpaint(img, mask, 6, cv2.INPAINT_TELEA)
    return dst

def apply_clahe(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    cl = clahe.apply(l)
    limg = cv2.merge((cl,a,b))
    return cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)

if os.path.exists(yolo_dataset_path):
    print("Membersihkan folder yolo_dataset sebelumnya...")
    shutil.rmtree(yolo_dataset_path)

for split in ['train', 'val', 'test']:
    for cls in classes:
        os.makedirs(os.path.join(yolo_dataset_path, split, cls), exist_ok=True)

train_df_final = train_df.copy()
if SKENARIO in [2, 5]:
    print("Menerapkan Weighted Random Sampling (Oversampling) pada Train set...")
    max_count = train_df['label'].value_counts().max()
    balanced_dfs = []
    
    for c in classes:
        df_c = train_df[train_df['dx'] == c]
        # Menduplikasi kelas minoritas agar setara dengan mayoritas
        df_c_oversampled = df_c.sample(max_count, replace=True, random_state=42)
        balanced_dfs.append(df_c_oversampled)
        
    train_df_final = pd.concat(balanced_dfs).reset_index(drop=True)

def process_and_save(dataframe, split_name):
    use_dullrazor = SKENARIO in [3, 5]
    use_clahe = SKENARIO in [4, 5]

    print(f"Memproses {len(dataframe)} gambar untuk {split_name}...")
    for idx, row in dataframe.iterrows():
        img_id = row['image_id']
        cls_name = row['dx']
        src_path = all_image_paths.get(img_id)

        if not src_path: continue

        # Tambahkan index ke nama file untuk menghindari overwrite jika WRS menduplikasi gambar
        dst_filename = f"{img_id}_{idx}.jpg" if split_name == 'train' and SKENARIO in [2, 5] else f"{img_id}.jpg"
        dst_path = os.path.join(yolo_dataset_path, split_name, cls_name, dst_filename)

        if use_dullrazor or use_clahe:
            img = cv2.imread(src_path)
            if img is None: continue
            
            if use_dullrazor:
                img = apply_dullrazor(img)
            if use_clahe:
                img = apply_clahe(img)
                
            cv2.imwrite(dst_path, img)
        else:
            if not os.path.exists(dst_path):
                shutil.copy2(src_path, dst_path)

process_and_save(train_df_final, 'train')
process_and_save(val_df, 'val')
process_and_save(test_df, 'test')

print("Struktur dataset YOLO siap digunakan untuk training")

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

print("Memulai proses training YOLOv8...")

model = YOLO('yolov8n-cls.pt')

# Memulai training
results = model.train(
    data=yolo_dataset_path, 
    epochs=15,             
    imgsz=224,            
    batch=32,              
    project='/content/drive/MyDrive/ML_Skin_Cancer/YOLO_Runs',
    name='baseline_yolov8',
    pretrained=True
)

In [ ]:
metrics = model.val(data=yolo_dataset_path, split='test')

Pakai device: cuda
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 108MB/s] 
